# Прогноз на следующий торговый день: SBER, OZON, VTBR

**Что делает этот ноутбук:**
1. Загружает исторические данные из БД
2. Walk-forward бэктест на последних **5 днях** (ML + DL + TA)
3. Получает новости за **сегодня** со smart-lab.ru
4. Обучает все модели на полной истории → прогноз на **12 мая 2026**

---
```bash
# Первый запуск:
make db-setup && make fetch-test DAYS=1000
```

## Шаг 1: Подготовка

In [67]:
import os
import sys
import re
import subprocess
import requests
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from datetime import datetime, date

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
PROJECT_ROOT = os.path.abspath('.')

load_dotenv()

def _env(key, default=''):
    val = os.getenv(key, default)
    return str(val).split('#')[0].strip().strip('"').strip("'").strip()

DB_HOST     = '127.0.0.1'
DB_PORT     = 5432
DB_NAME     = _env('DB_NAME', 'postgres')
DB_USER     = _env('DB_USER', 'postgres')
DB_PASSWORD = _env('DB_PASSWORD', 'postgres')

DATABASE_URL = f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(DATABASE_URL)

TODAY         = '2026-05-11'
FORECAST_DATE = '2026-05-12'
LAST_N_DAYS   = 5   # walk-forward глубина бэктеста

print(f'БД: {DB_HOST}:{DB_PORT}/{DB_NAME}')
print(f'Сегодня: {TODAY}  |  Прогноз на: {FORECAST_DATE}')
print(f'Walk-forward: последние {LAST_N_DAYS} дней')

БД: 127.0.0.1:5432/postgres
Сегодня: 2026-05-11  |  Прогноз на: 2026-05-12
Walk-forward: последние 5 дней


## Шаг 2: ML и DL модели

In [68]:
from models.ridge import RidgeTradeModel
from models.xgboost_model import XGBoostTradeModelNew
from models.lightgbm_model import LightGBMTradeModelNew
from models.cat_boost_model import CatBoostTradeModelNew
from models.random_forest_regression_model import RandomForestTradeModelNew
from models.rf_classifier import RandomForestClassifierNew

ML_MODELS = {
    'ridge':         RidgeTradeModel,
    'xgboost':       XGBoostTradeModelNew,
    'lightgbm':      LightGBMTradeModelNew,
    'catboost':      CatBoostTradeModelNew,
    'random_forest': RandomForestTradeModelNew,
    'rf_classifier': RandomForestClassifierNew,
}

# DL модели запускаются через subprocess (как в walkforward_ta_ml_dl.ipynb)
# DL_MODELS = ['lstm', 'tcn']
DL_MODELS = ['tcn']


print(f'ML: {list(ML_MODELS.keys())}')
print(f'DL: {DL_MODELS}  (запуск через subprocess)')

ML: ['ridge', 'xgboost', 'lightgbm', 'catboost', 'random_forest', 'rf_classifier']
DL: ['tcn']  (запуск через subprocess)


## Шаг 3: Загрузка данных

In [69]:
TEST_TICKERS = ['SBER', 'OZON', 'VTBR']

tickers_str = "', '".join(TEST_TICKERS)
with engine.connect() as conn:
    tickers_df = pd.read_sql(
        text(f"SELECT ticker, figi FROM public.tickers WHERE ticker IN ('{tickers_str}')"),
        conn
    )

if tickers_df.empty:
    print('❌ Тикеры не найдены. Запустите: make fetch-test')
    raise SystemExit(1)

ticker_to_figi = dict(zip(tickers_df['ticker'], tickers_df['figi']))
print('Тикеры:', ticker_to_figi)

def load_candles_from_db(figi):
    q = text(f'SELECT timestamp, open, high, low, close, volume FROM all_dfs."{figi}" ORDER BY timestamp')
    df = pd.read_sql(q, engine)
    if not df.empty:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df

candles_data = {}
for ticker, figi in ticker_to_figi.items():
    df = load_candles_from_db(figi)
    if not df.empty:
        candles_data[ticker] = df
        print(f'{ticker}: {len(df)} свечей, последняя: {df["timestamp"].max().strftime("%Y-%m-%d")}')
    else:
        print(f'{ticker}: ❌ нет данных — запустите make fetch-test DAYS=1000')

if not candles_data:
    raise SystemExit(1)

Тикеры: {'SBER': 'BBG004730N88', 'OZON': 'TCS80A10CW95', 'VTBR': 'BBG004730ZJ9'}
SBER: 789 свечей, последняя: 2026-05-15
OZON: 698 свечей, последняя: 2026-05-15
VTBR: 785 свечей, последняя: 2026-05-15


## Шаг 4: TA-индикаторы (расширенный набор)

In [70]:
def calculate_indicators(df):
    df = df.copy()

    # RSI
    delta = df['close'].diff()
    gain  = delta.where(delta > 0, 0).rolling(14).mean()
    loss  = (-delta.where(delta < 0, 0)).rolling(14).mean()
    df['rsi'] = 100 - (100 / (1 + np.where(loss != 0, gain / loss, 100)))

    # MACD
    ema12 = df['close'].ewm(span=12, adjust=False).mean()
    ema26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd']        = ema12 - ema26
    df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_hist']   = df['macd'] - df['macd_signal']

    # SMA / EMA
    for w in [5, 20, 50]:
        df[f'sma_{w}'] = df['close'].rolling(w).mean()
        df[f'ema_{w}'] = df['close'].ewm(span=w, adjust=False).mean()

    # Bollinger Bands
    bb_mid          = df['close'].rolling(20).mean()
    bb_std          = df['close'].rolling(20).std()
    df['bb_upper']  = bb_mid + bb_std * 2
    df['bb_lower']  = bb_mid - bb_std * 2
    df['bb_position'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])

    # Stochastic %K
    low14  = df['low'].rolling(14).min()
    high14 = df['high'].rolling(14).max()
    df['stoch_k'] = 100 * ((df['close'] - low14) / (high14 - low14))
    df['stoch_d'] = df['stoch_k'].rolling(3).mean()

    # Williams %R
    df['williams_r'] = -100 * ((high14 - df['close']) / (high14 - low14))

    # ATR
    tr = pd.concat([
        df['high'] - df['low'],
        (df['high'] - df['close'].shift()).abs(),
        (df['low']  - df['close'].shift()).abs()
    ], axis=1).max(axis=1)
    df['atr14'] = tr.rolling(14).mean()

    # ADX
    plus_dm  = df['high'].diff().clip(lower=0)
    minus_dm = (-df['low'].diff()).clip(lower=0)
    plus_dm[plus_dm < minus_dm]  = 0
    minus_dm[minus_dm < plus_dm] = 0
    smooth_tr   = tr.rolling(14).sum()
    plus_di14   = 100 * plus_dm.rolling(14).sum()  / (smooth_tr + 1e-9)
    minus_di14  = 100 * minus_dm.rolling(14).sum() / (smooth_tr + 1e-9)
    dx          = 100 * (plus_di14 - minus_di14).abs() / (plus_di14 + minus_di14 + 1e-9)
    df['adx14'] = dx.rolling(14).mean()

    # OBV
    obv = np.where(df['close'] > df['close'].shift(), df['volume'],
           np.where(df['close'] < df['close'].shift(), -df['volume'], 0))
    df['obv'] = obv.cumsum()

    # CCI
    tp           = (df['high'] + df['low'] + df['close']) / 3
    sma_tp       = tp.rolling(20).mean()
    mad          = tp.rolling(20).apply(lambda x: np.abs(x - x.mean()).mean())
    df['cci20']  = (tp - sma_tp) / (0.015 * mad + 1e-9)

    # Price vs SMA
    df['price_vs_sma20'] = df['close'] / df['sma_20'] - 1
    df['next_return']    = df['close'].shift(-1) / df['close'] - 1

    return df


def generate_ta_signals(df):
    df = df.copy()

    # RSI
    df['signal_rsi'] = 0
    df.loc[df['rsi'] < 30, 'signal_rsi'] = 1
    df.loc[df['rsi'] > 70, 'signal_rsi'] = -1

    # MACD cross
    mp, ms_p = df['macd'].shift(1), df['macd_signal'].shift(1)
    df['signal_macd'] = 0
    df.loc[(df['macd'] > df['macd_signal']) & (mp <= ms_p), 'signal_macd'] = 1
    df.loc[(df['macd'] < df['macd_signal']) & (mp >= ms_p), 'signal_macd'] = -1

    # SMA 5/20 cross
    s5p, s20p = df['sma_5'].shift(1), df['sma_20'].shift(1)
    df['signal_sma_cross'] = 0
    df.loc[(df['sma_5'] > df['sma_20']) & (s5p <= s20p), 'signal_sma_cross'] = 1
    df.loc[(df['sma_5'] < df['sma_20']) & (s5p >= s20p), 'signal_sma_cross'] = -1

    # Trend (price vs SMA20)
    df['signal_trend'] = 0
    df.loc[df['price_vs_sma20'] > 0.02,  'signal_trend'] = 1
    df.loc[df['price_vs_sma20'] < -0.02, 'signal_trend'] = -1

    # Bollinger
    df['signal_bb'] = 0
    df.loc[df['bb_position'] < 0.1, 'signal_bb'] = 1
    df.loc[df['bb_position'] > 0.9, 'signal_bb'] = -1

    # Stochastic
    df['signal_stoch'] = 0
    df.loc[df['stoch_k'] < 20, 'signal_stoch'] = 1
    df.loc[df['stoch_k'] > 80, 'signal_stoch'] = -1

    # Williams %R
    df['signal_williams'] = 0
    df.loc[df['williams_r'] < -80, 'signal_williams'] = 1
    df.loc[df['williams_r'] > -20, 'signal_williams'] = -1

    # CCI
    df['signal_cci'] = 0
    df.loc[df['cci20'] < -100, 'signal_cci'] = 1
    df.loc[df['cci20'] >  100, 'signal_cci'] = -1

    return df


SIGNAL_COLS = ['signal_rsi', 'signal_macd', 'signal_sma_cross', 'signal_trend',
               'signal_bb', 'signal_stoch', 'signal_williams', 'signal_cci']
IND_NAMES   = {'signal_rsi': 'RSI', 'signal_macd': 'MACD', 'signal_sma_cross': 'SMA',
               'signal_trend': 'Trend', 'signal_bb': 'BB', 'signal_stoch': 'Stoch',
               'signal_williams': 'Williams%R', 'signal_cci': 'CCI'}

for ticker in candles_data:
    candles_data[ticker] = calculate_indicators(candles_data[ticker])
    candles_data[ticker] = generate_ta_signals(candles_data[ticker])

print(f'TA-индикаторы: RSI, MACD, SMA, BB, Stoch, Williams%R, ADX, ATR, OBV, CCI')
print(f'TA-сигналы: {list(IND_NAMES.values())}')

TA-индикаторы: RSI, MACD, SMA, BB, Stoch, Williams%R, ADX, ATR, OBV, CCI
TA-сигналы: ['RSI', 'MACD', 'SMA', 'Trend', 'BB', 'Stoch', 'Williams%R', 'CCI']


## Шаг 5: Вспомогательные функции (ML / DL)

In [71]:
S_MAP = {'BUY': 1, 'SELL': -1, 'HOLD': 0, 'NEUTRAL': 0}


def train_ml_models(train_df, test_df):
    """Обучает ML модели, возвращает dict[name] = {signal, r2, direction_accuracy, predicted_price, current_price, expected_change}."""
    results = {}
    if len(train_df) < 100:
        return results
    for name, ModelClass in ML_MODELS.items():
        try:
            model = ModelClass(test_size=0.2, random_state=42)
            model.train(train_df)
            pred = model.predict_next(test_df)
            m    = model.test_metrics or {}
            results[name] = {
                'signal':            S_MAP.get(pred.get('signal', 'HOLD'), 0),
                'signal_str':        pred.get('signal', 'HOLD'),
                'predicted_price':   pred.get('predicted_price'),
                'current_price':     pred.get('current_price'),
                'expected_change':   pred.get('expected_change'),
                'r2':                m.get('test_r2', 0) or 0,
                'direction_accuracy': m.get('test_direction_accuracy', 50) or 50,
            }
        except Exception as e:
            results[name] = {'signal': 0, 'signal_str': 'ERR', 'error': str(e)}
    return results


def get_dl_predictions(figi):
    """Запускает LSTM и TCN через subprocess, возвращает dict."""
    results = {}
    for model_name in DL_MODELS:
        try:
            script = f'''
import sys
sys.path.insert(0, "{PROJECT_ROOT}")
import os
os.environ["DB_HOST"] = "{DB_HOST}"
os.environ["DB_PORT"] = "{DB_PORT}"
os.environ["DB_NAME"] = "{DB_NAME}"
os.environ["DB_USER"] = "{DB_USER}"
os.environ["DB_PASSWORD"] = "{DB_PASSWORD}"
from models.{model_name}_model import main
main("{figi}")
'''
            result = subprocess.run(
                ['python', '-c', script],
                capture_output=True, text=True, timeout=300, cwd=PROJECT_ROOT
            )
            out = result.stdout + result.stderr
            sm  = re.search(r'Торговый сигнал:\s*(\w+)', out)
            r2m = re.search(r'R²:\s*([\-\d.]+)', out)
            dm  = re.search(r'Direction Accuracy:\s*([\d.]+)', out)
            pm  = re.search(r'Прогнозируемая цена:\s*([\d.]+)', out)
            cm  = re.search(r'Текущая цена:\s*([\d.]+)', out)
            em  = re.search(r'Ожидаемое изменение:\s*([\+\-\d.]+)%', out)
            if sm:
                results[model_name] = {
                    'signal':            S_MAP.get(sm.group(1), 0),
                    'signal_str':        sm.group(1),
                    'r2':                float(r2m.group(1)) if r2m else 0,
                    'direction_accuracy': float(dm.group(1)) if dm else 50,
                    'predicted_price':   float(pm.group(1)) if pm else None,
                    'current_price':     float(cm.group(1)) if cm else None,
                    'expected_change':   float(em.group(1)) if em else None,
                }
        except Exception as e:
            results[model_name] = {'signal': 0, 'signal_str': 'ERR', 'error': str(e)}
    return results


def get_ml_consensus(all_model_results, min_models=2):
    """Возвращает {'signal': 1/-1/0, 'pct': float}."""
    sigs = [v['signal'] for v in all_model_results.values()]
    if len(sigs) < min_models:
        return {'signal': 0, 'pct': 0}
    buy  = sigs.count(1)
    sell = sigs.count(-1)
    n    = len(sigs)
    if buy > n / 2:
        return {'signal': 1,  'pct': buy / n * 100}
    if sell > n / 2:
        return {'signal': -1, 'pct': sell / n * 100}
    return {'signal': 0, 'pct': max(buy, sell) / n * 100}


def print_model_results(ticker, ml_results, dl_results=None, title=''):
    """Красиво выводит результаты всех моделей для тикера."""
    all_res = {**ml_results, **(dl_results or {})}
    if title:
        print(f'\n  {title}')
    print(f'  {"Модель":<18} {"Сигнал":<6} {"Прогноз":>10} {"Изменение":>10} {"R²":>7} {"Dir%":>7}')
    print(f'  {"─"*65}')
    for name, r in all_res.items():
        if r.get('signal_str') == 'ERR':
            print(f'  {name:<18} ❌ {r.get("error", "")[:30]}')
            continue
        sig   = r.get('signal_str', 'HOLD')
        pred  = f'{r["predicted_price"]:.2f}' if r.get('predicted_price') else '—'
        chg   = f'{r["expected_change"]:+.2f}%' if r.get('expected_change') is not None else '—'
        r2    = f'{r["r2"]:.3f}' if r.get('r2') is not None else '—'
        dacc  = f'{r["direction_accuracy"]:.1f}' if r.get('direction_accuracy') is not None else '—'
        mark  = '★' if sig == 'BUY' else ('▼' if sig == 'SELL' else ' ')
        print(f'  {name:<18} {mark}{sig:<5} {pred:>10} {chg:>10} {r2:>7} {dacc:>7}')


print('Функции готовы: train_ml_models, get_dl_predictions, get_ml_consensus, print_model_results')

Функции готовы: train_ml_models, get_dl_predictions, get_ml_consensus, print_model_results


## Шаг 6: Новости за сегодня (2026-05-11)

In [54]:
NEWS_CACHE = {}

COMPANY_SEARCH = {'SBER': 'сбер', 'OZON': 'ozon', 'VTBR': 'втб'}
MONTH_MAP = {
    'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
    'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
    'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12
}


def get_news_by_date(ticker, target_date, max_pages=30):
    """Получает новости со smart-lab.ru за конкретную дату."""
    if isinstance(target_date, str):
        target_date = datetime.strptime(target_date, '%Y-%m-%d').date()

    cache_key = (ticker, str(target_date))
    if cache_key in NEWS_CACHE:
        return NEWS_CACHE[cache_key]

    search  = COMPANY_SEARCH.get(ticker, ticker.lower())
    headers = {'User-Agent': 'Mozilla/5.0'}
    found   = []

    for page in range(1, max_pages + 1):
        url = (f'https://smart-lab.ru/search/topics/?blog=news&q={search}' if page == 1
               else f'https://smart-lab.ru/search/topics/page{page}/?q={search}&blog=news')
        try:
            resp  = requests.get(url, headers=headers, timeout=15)
            soup  = BeautifulSoup(resp.content, 'lxml')
            topics = soup.find_all('div', class_=lambda x: x and 'topic' in x)
            if not topics:
                break
            for t in topics:
                h2 = t.find('h2', class_='title')
                if not h2:
                    continue
                a = h2.find('a')
                if not a:
                    continue
                title    = a.get('title') or a.get_text(strip=True)
                date_el  = t.find('li', class_='date')
                date_str = date_el.get_text(strip=True)[:20] if date_el else ''
                pub_date = None
                try:
                    parts = date_str.replace(',', '').split()
                    if len(parts) >= 3:
                        pub_date = date(int(parts[2]), MONTH_MAP.get(parts[1], 1), int(parts[0]))
                except Exception:
                    pass
                if pub_date == target_date:
                    found.append({'date': date_str, 'title': title})
            if len(found) >= 5:
                break
        except Exception:
            break

    NEWS_CACHE[cache_key] = found
    return found


print(f'Новости за {TODAY}:\n')
today_news = {}
for ticker in TEST_TICKERS:
    news = get_news_by_date(ticker, TODAY)
    today_news[ticker] = news
    print(f'  {ticker}: {len(news)} новостей')
    for n in news[:3]:
        print(f'    • {n["title"][:90]}')
    if not news:
        print(f'    — новостей не найдено')

Новости за 2026-05-11:

  SBER: 0 новостей
    — новостей не найдено
  OZON: 0 новостей
    — новостей не найдено
  VTBR: 0 новостей
    — новостей не найдено


## Шаг 7: Walk-Forward — последние 5 дней

In [55]:
wf_summary = []   # для итоговой таблицы

for ticker, df in candles_data.items():
    figi = ticker_to_figi[ticker]
    n    = len(df)

    print(f'\n{"="*65}')
    print(f'  WALK-FORWARD [{ticker}]  |  {n} свечей  |  последние {LAST_N_DAYS} дней')
    print(f'{"="*65}')

    if n < 100 + LAST_N_DAYS:
        print('  ❌ Мало данных')
        continue

    test_indices     = range(n - LAST_N_DAYS - 1, n - 1)
    dl_results_cache = {}

    for i in test_indices:
        train_df      = df.iloc[:i].copy()
        test_df       = df.iloc[:i+1].copy()
        pred_date     = df.iloc[i + 1]['timestamp'].strftime('%Y-%m-%d') if i + 1 < n else '?'
        actual_close  = df.iloc[i + 1]['close'] if i + 1 < n else None
        current_close = df.iloc[i]['close']

        actual_str = f'{actual_close:.2f}' if actual_close is not None else '—'
        print(f'\n  ── Прогноз на {pred_date}  (текущий: {current_close:.2f},  фактический: {actual_str})')

        # ML
        ml_res = train_ml_models(train_df, test_df)

        # DL — один раз, затем кэш
        if not dl_results_cache:
            print('  [DL] запуск lstm/tcn...')
            dl_results_cache = get_dl_predictions(figi)
        dl_res = dl_results_cache

        all_res   = {**ml_res, **dl_res}
        consensus = get_ml_consensus(all_res)
        cons_str  = {1: 'BUY', -1: 'SELL', 0: 'HOLD'}[consensus['signal']]

        # TA текущего дня
        row      = df.iloc[i]
        ta_buy   = sum(1 for c in SIGNAL_COLS if row.get(c, 0) == 1)
        ta_sell  = sum(1 for c in SIGNAL_COLS if row.get(c, 0) == -1)
        ta_str   = 'BUY' if ta_buy > ta_sell else ('SELL' if ta_sell > ta_buy else 'HOLD')

        print_model_results(ticker, ml_res, dl_res)

        # Точность прогноза
        if actual_close is not None:
            errors = {
                name: abs(r['predicted_price'] - actual_close) / actual_close * 100
                for name, r in all_res.items()
                if r.get('predicted_price')
            }
            if errors:
                best    = min(errors, key=errors.get)
                worst   = max(errors, key=errors.get)
                avg_err = np.mean(list(errors.values()))
                print(f'\n  Ошибка:  avg={avg_err:.2f}%  '
                      f'лучший={best}({errors[best]:.2f}%)  '
                      f'худший={worst}({errors[worst]:.2f}%)')

        print(f'  ML-консенсус: {cons_str} ({consensus["pct"]:.0f}%)  |  TA: {ta_str} (BUY:{ta_buy} SELL:{ta_sell})')

        for name, r in all_res.items():
            err = (abs(r['predicted_price'] - actual_close) / actual_close * 100
                   if r.get('predicted_price') and actual_close is not None else None)
            wf_summary.append({
                'ticker': ticker, 'model': name, 'pred_date': pred_date,
                'current': current_close, 'actual': actual_close,
                'predicted': r.get('predicted_price'),
                'signal': r.get('signal_str', '?'),
                'r2': r.get('r2'), 'dir_acc': r.get('direction_accuracy'),
                'error_pct': err,
            })

print('\n✅ Walk-forward завершён')


  WALK-FORWARD [SBER]  |  788 свечей  |  последние 5 дней

  ── Прогноз на 2026-05-07  (текущий: 319.90,  фактический: 318.77)
  [DL] запуск lstm/tcn...
  Модель             Сигнал    Прогноз  Изменение      R²    Dir%
  ─────────────────────────────────────────────────────────────────
  ridge               HOLD      318.70     -0.38%   0.998    90.4
  xgboost            ▼SELL      316.47     -1.07%   0.984    84.1
  lightgbm           ★BUY       322.11     +0.69%   0.986    82.2
  catboost           ▼SELL      314.04     -1.83%   0.966    75.2
  random_forest       NEUTRAL     320.12     +0.07%   0.985    78.3
  rf_classifier      ❌ 'RandomForestClassifierNew' ob
  tcn                ▼SELL      296.65     -8.93%   0.691    56.5

  Ошибка:  avg=1.77%  лучший=ridge(0.02%)  худший=tcn(6.94%)
  ML-консенсус: HOLD (43%)  |  TA: BUY (BUY:1 SELL:0)

  ── Прогноз на 2026-05-08  (текущий: 318.77,  фактический: 320.39)
  Модель             Сигнал    Прогноз  Изменение      R²    Dir%
  ───────

KeyboardInterrupt: 

## Шаг 7b: Таблица точности по walk-forward

In [36]:
if wf_summary:
    wf_df = pd.DataFrame(wf_summary).dropna(subset=['error_pct'])
    acc_table = (
        wf_df.groupby(['ticker', 'model'])
        .agg(avg_error=('error_pct', 'mean'),
             avg_r2    =('r2', 'mean'),
             avg_dir   =('dir_acc', 'mean'))
        .round(3)
        .sort_values(['ticker', 'avg_error'])
    )
    print('Walk-forward точность (средняя ошибка за 5 дней):')
    display(acc_table)

Walk-forward точность (средняя ошибка за 5 дней):


avg_error  avg_r2  avg_dir
ticker model                                    
OZON   ridge              0.255   0.993   90.935
       lightgbm           0.768   0.665   73.669
       random_forest      0.800   0.645   64.317
       xgboost            1.024   0.588   62.014
       catboost           1.068   0.248   54.676
       lstm               4.110   0.000   50.000
       tcn                6.355   0.921   57.934
SBER   ridge              0.120   0.998   90.840
       random_forest      0.465   0.985   80.154
       tcn                0.472   0.607   55.772
       lightgbm           0.534   0.986   80.535
       xgboost            1.144   0.978   81.425
       catboost           1.428   0.972   77.100
       lstm               3.790   0.000   50.000
VTBR   ridge              0.070   0.998   87.978
       xgboost            0.230   0.990   69.303
       random_forest      0.352   0.989   67.135
       lightgbm           0.426   0.991   71.988
       catboost           0.868   0.990   76.594
       lstm               3.694   0.000   50.000
       tcn                9.583   0.816   58.006

## Шаг 8: Прогноз на 2026-05-15 (обучение на полной истории)

In [72]:
forecast_results = []
dl_forecast_cache = {}

for ticker, df in candles_data.items():
    figi     = ticker_to_figi[ticker]
    last_row = df.iloc[-1]

    print(f'\n{"="*65}')
    print(f'  ПРОГНОЗ НА {FORECAST_DATE}  |  {ticker}  |  {len(df)} свечей')
    print(f'  Последняя свеча: {last_row["timestamp"].strftime("%Y-%m-%d")}  |  close={last_row["close"]:.2f}')
    print(f'{"="*65}')

    # ─── TA: подробный вывод ────────────────────────────────────────
    def _safe(v):
        return float(v) if v is not None and not (isinstance(v, float) and np.isnan(v)) else None

    rsi     = _safe(last_row.get('rsi'))
    macd    = _safe(last_row.get('macd'))
    macd_sl = _safe(last_row.get('macd_signal'))
    macd_h  = _safe(last_row.get('macd_hist'))
    sma5    = _safe(last_row.get('sma_5'))
    sma20   = _safe(last_row.get('sma_20'))
    bb_pos  = _safe(last_row.get('bb_position'))
    bb_up   = _safe(last_row.get('bb_upper'))
    bb_lo   = _safe(last_row.get('bb_lower'))
    stoch   = _safe(last_row.get('stoch_k'))
    will    = _safe(last_row.get('williams_r'))
    cci     = _safe(last_row.get('cci20'))
    adx     = _safe(last_row.get('adx14'))
    atr     = _safe(last_row.get('atr14'))

    def sig_label(cond_buy, cond_sell):
        if cond_buy:   return '★ BUY'
        if cond_sell:  return '▼ SELL'
        return '  нейтр.'

    ta_rows = [
        ('RSI(14)',     f'{rsi:.1f}' if rsi else '—',
         sig_label(rsi is not None and rsi < 30, rsi is not None and rsi > 70),
         f'зоны: <30 BUY | >70 SELL'),
        ('MACD',        f'{macd:.3f}' if macd else '—',
         sig_label(macd is not None and macd_sl is not None and macd > macd_sl,
                   macd is not None and macd_sl is not None and macd < macd_sl),
         f'signal={macd_sl:.3f}, hist={macd_h:.3f}' if macd_sl and macd_h else ''),
        ('SMA 5/20',    f'{sma5:.2f}' if sma5 else '—',
         sig_label(sma5 is not None and sma20 is not None and sma5 > sma20,
                   sma5 is not None and sma20 is not None and sma5 < sma20),
         f'SMA20={sma20:.2f}' if sma20 else ''),
        ('Bollinger',   f'{bb_pos:.2f}' if bb_pos is not None else '—',
         sig_label(bb_pos is not None and bb_pos < 0.1,
                   bb_pos is not None and bb_pos > 0.9),
         f'диапазон [{bb_lo:.2f} – {bb_up:.2f}]' if bb_lo and bb_up else ''),
        ('Stoch %K',    f'{stoch:.1f}' if stoch else '—',
         sig_label(stoch is not None and stoch < 20,
                   stoch is not None and stoch > 80),
         'зоны: <20 BUY | >80 SELL'),
        ('Williams %R', f'{will:.1f}' if will else '—',
         sig_label(will is not None and will < -80,
                   will is not None and will > -20),
         'зоны: <-80 BUY | >-20 SELL'),
        ('CCI(20)',     f'{cci:.1f}' if cci else '—',
         sig_label(cci is not None and cci < -100,
                   cci is not None and cci > 100),
         'зоны: <-100 BUY | >100 SELL'),
        ('ADX(14)',     f'{adx:.1f}' if adx else '—',
         '  сильный' if adx and adx > 25 else '  слабый',
         '>25 = сильный тренд'),
        ('ATR(14)',     f'{atr:.2f}' if atr else '—', '', 'волатильность'),
    ]

    print(f'\n  {"Индикатор":<14} {"Значение":>10}  {"Сигнал":<12}  Детали')
    print(f'  {"─"*62}')
    for name, val, sig, detail in ta_rows:
        print(f'  {name:<14} {val:>10}  {sig:<12}  {detail}')

    # Считаем TA-консенсус по текущему состоянию (не crossover)
    ta_state_signals = [
        1  if rsi    is not None and rsi < 30    else (-1 if rsi    is not None and rsi > 70     else 0),
        1  if (macd  is not None and macd_sl is not None and macd > macd_sl) else
        (-1 if (macd is not None and macd_sl is not None and macd < macd_sl) else 0),
        1  if (sma5  is not None and sma20 is not None and sma5 > sma20) else
        (-1 if (sma5 is not None and sma20 is not None and sma5 < sma20) else 0),
        1  if bb_pos is not None and bb_pos < 0.1  else (-1 if bb_pos is not None and bb_pos > 0.9  else 0),
        1  if stoch  is not None and stoch < 20     else (-1 if stoch  is not None and stoch > 80    else 0),
        1  if will   is not None and will < -80      else (-1 if will   is not None and will > -20    else 0),
        1  if cci    is not None and cci < -100      else (-1 if cci    is not None and cci > 100     else 0),
    ]
    ta_buy  = ta_state_signals.count(1)
    ta_sell = ta_state_signals.count(-1)
    ta_str  = 'BUY' if ta_buy > ta_sell else ('SELL' if ta_sell > ta_buy else 'HOLD')
    print(f'\n  TA-консенсус (состояние): {ta_str}  (BUY:{ta_buy} SELL:{ta_sell} нейтр.:{ta_state_signals.count(0)})')

    # ─── ML ────────────────────────────────────────────────────────
    ml_res = train_ml_models(df, df)

    # ─── DL ────────────────────────────────────────────────────────
    if ticker not in dl_forecast_cache:
        print('\n  [DL] запуск lstm/tcn...')
        dl_forecast_cache[ticker] = get_dl_predictions(figi)
    dl_res = dl_forecast_cache[ticker]

    all_res   = {**ml_res, **dl_res}
    consensus = get_ml_consensus(all_res)
    cons_str  = {1: 'BUY', -1: 'SELL', 0: 'HOLD'}[consensus['signal']]

    print_model_results(ticker, ml_res, dl_res,
                        title=f'Результаты моделей → прогноз на {FORECAST_DATE}')

    # ─── Итог ──────────────────────────────────────────────────────
    ta_int   = 1 if ta_buy > ta_sell else (-1 if ta_sell > ta_buy else 0)
    ml_int   = consensus['signal']
    combined = ta_int + ml_int
    final    = 'BUY' if combined > 0 else ('SELL' if combined < 0 else 'HOLD')

    prices   = [r['predicted_price'] for r in all_res.values() if r.get('predicted_price')]
    avg_pred = np.mean(prices) if prices else None
    curr     = last_row['close']

    news = today_news.get(ticker, [])
    if news:
        print(f'\n  Новости {TODAY}:')
        for n in news[:3]:
            print(f'    • {n["title"][:85]}')

    print(f'\n  ML-консенсус: {cons_str} ({consensus["pct"]:.0f}%)  |  TA: {ta_str}  →  ИТОГ: {final}')
    if avg_pred:
        print(f'  Средний прогноз: {avg_pred:.2f}  (текущая: {curr:.2f},  изм.: {(avg_pred/curr-1)*100:+.2f}%)')

    for name, r in all_res.items():
        forecast_results.append({
            'ticker': ticker, 'model': name,
            'forecast_date': FORECAST_DATE,
            'current_price':   r.get('current_price', curr),
            'predicted_price': r.get('predicted_price'),
            'expected_change': r.get('expected_change'),
            'signal':          r.get('signal_str', '?'),
            'r2':              r.get('r2'),
            'dir_acc':         r.get('direction_accuracy'),
            'ta_signal':       ta_str,
            'final_signal':    final,
        })

print('\n✅ Прогнозы получены')


  ПРОГНОЗ НА 2026-05-12  |  SBER  |  789 свечей
  Последняя свеча: 2026-05-15  |  close=322.98

  Индикатор        Значение  Сигнал        Детали
  ──────────────────────────────────────────────────────────────
  RSI(14)              58.3    нейтр.      зоны: <30 BUY | >70 SELL
  MACD                0.869  ★ BUY         signal=0.681, hist=0.188
  SMA 5/20           324.73  ★ BUY         SMA20=322.26
  Bollinger            0.57    нейтр.      диапазон [317.32 – 327.20]
  Stoch %K             54.2    нейтр.      зоны: <20 BUY | >80 SELL
  Williams %R         -45.8    нейтр.      зоны: <-80 BUY | >-20 SELL
  CCI(20)              39.3    нейтр.      зоны: <-100 BUY | >100 SELL
  ADX(14)              36.0    сильный     >25 = сильный тренд
  ATR(14)              3.01                волатильность

  TA-консенсус (состояние): BUY  (BUY:2 SELL:0 нейтр.:5)

  [DL] запуск lstm/tcn...

  Результаты моделей → прогноз на 2026-05-12
  Модель             Сигнал    Прогноз  Изменение      R²    Dir%


## Шаг 9: Сводная таблица прогнозов

In [73]:
if forecast_results:
    df_fc = pd.DataFrame(forecast_results)

    fmt = df_fc.copy()
    fmt['predicted_price'] = fmt['predicted_price'].apply(lambda v: f'{v:.2f}' if v else '—')
    fmt['expected_change'] = fmt['expected_change'].apply(lambda v: f'{v:+.2f}%' if v is not None else '—')
    fmt['r2']              = fmt['r2'].apply(lambda v: f'{v:.3f}' if v is not None else '—')
    fmt['dir_acc']         = fmt['dir_acc'].apply(lambda v: f'{v:.1f}%' if v is not None else '—')
    fmt['current_price']   = fmt['current_price'].apply(lambda v: f'{v:.2f}' if v else '—')

    fmt = fmt.rename(columns={
        'ticker': 'Тикер', 'model': 'Модель', 'forecast_date': 'Дата',
        'current_price': 'Цена', 'predicted_price': f'Прогноз',
        'expected_change': 'Изм.', 'signal': 'ML сигнал',
        'r2': 'R²', 'dir_acc': 'Dir.Acc.',
        'ta_signal': 'TA', 'final_signal': '→ Итог',
    })
    display(fmt.reset_index(drop=True))

,Тикер,Модель,Дата,Цена,Прогноз,Изм.,ML сигнал,R²,Dir.Acc.,TA,→ Итог
0,SBER,ridge,2026-05-12,322.98,321.56,-0.44%,HOLD,0.998,92.4%,BUY,BUY
1,SBER,xgboost,2026-05-12,322.98,320.49,-0.77%,SELL,0.972,82.9%,BUY,BUY
2,SBER,lightgbm,2026-05-12,322.98,324.91,+0.60%,BUY,0.986,82.3%,BUY,BUY
3,SBER,catboost,2026-05-12,322.98,317.69,-1.64%,SELL,0.957,77.8%,BUY,BUY
4,SBER,random_forest,2026-05-12,322.98,323.56,+0.18%,HOLD,0.986,82.3%,BUY,BUY
5,SBER,rf_classifier,2026-05-12,322.98,nan,+nan%,ERR,nan,nan%,BUY,BUY
6,SBER,tcn,2026-05-12,322.98,290.68,-10.00%,SELL,0.539,53.8%,BUY,BUY
7,OZON,ridge,2026-05-12,4117.50,4065.55,-1.26%,SELL,0.993,91.4%,BUY,HOLD
8,OZON,xgboost,2026-05-12,4117.50,4082.86,-0.84%,SELL,0.549,62.9%,BUY,HOLD
9,OZON,lightgbm,2026-05-12,4117.50,4079.41,-0.93%,SELL,0.563,74.3%,BUY,HOLD


## Шаг 10: Консенсус по тикерам

In [74]:
if forecast_results:
    df_res = pd.DataFrame(forecast_results).dropna(subset=['predicted_price'])

    cons = (
        df_res.groupby('ticker')
        .agg(
            current_price   =('current_price',   'first'),
            avg_predicted   =('predicted_price', 'mean'),
            avg_change      =('expected_change', 'mean'),
            models_count    =('model', 'count'),
            buy_count       =('signal', lambda s: (s.str.upper() == 'BUY').sum()),
            sell_count      =('signal', lambda s: (s.str.upper() == 'SELL').sum()),
            ta_signal       =('ta_signal', 'first'),
            final_signal    =('final_signal', 'first'),
        ).reset_index()
    )

    print(f'\n{"╔" + "═"*62 + "╗"}')
    print(f'║  КОНСЕНСУС-ПРОГНОЗ НА {FORECAST_DATE}{" "*37}║')
    print(f'{"╠" + "═"*62 + "╣"}')
    print(f'║  {"Тикер":<7} {"Цена":>8} {"Прогноз":>9} {"Изм.":>8} {"ML":>5} {"TA":>5} {"Итог":>6} {"BUY/SELL/N":>12}  ║')
    print(f'{"╠" + "═"*62 + "╣"}')

    for _, row in cons.iterrows():
        sign = '+' if (row['avg_change'] or 0) >= 0 else ''
        ml_s = 'BUY' if row['buy_count'] > row['sell_count'] else ('SELL' if row['sell_count'] > row['buy_count'] else 'HOLD')
        votes = f'{int(row["buy_count"])}/{int(row["sell_count"])}/{int(row["models_count"]-row["buy_count"]-row["sell_count"])}'
        print(f'║  {row["ticker"]:<7} {row["current_price"]:>8.2f} {row["avg_predicted"]:>9.2f} '
              f'{sign}{row["avg_change"]:>7.2f}% '
              f'{ml_s:>5} {row["ta_signal"]:>5} {row["final_signal"]:>6} {votes:>12}  ║')

    print(f'{"╚" + "═"*62 + "╝"}')
    print(f'   BUY/SELL/N = количество моделей с сигналом BUY / SELL / нейтральных')


╔══════════════════════════════════════════════════════════════╗
║  КОНСЕНСУС-ПРОГНОЗ НА 2026-05-12                                     ║
╠══════════════════════════════════════════════════════════════╣
║  Тикер       Цена   Прогноз     Изм.    ML    TA   Итог   BUY/SELL/N  ║
╠══════════════════════════════════════════════════════════════╣
║  OZON     4117.50   4005.31   -2.72%  SELL   BUY   HOLD        0/6/0  ║
║  SBER      322.98    316.48   -2.01%  SELL   BUY    BUY        1/3/2  ║
║  VTBR       89.79     88.15   -1.82%  SELL   BUY    BUY        1/3/2  ║
╚══════════════════════════════════════════════════════════════╝
   BUY/SELL/N = количество моделей с сигналом BUY / SELL / нейтральных
